# 08 · Test of H2: does a consistent grasp close the gap to teleop?

**Hypothesis H2** (README, "Why is the policy trained on my demos worse"): my 10 training demos grasp the rim at different places (86-149° around the bowl), so for the same image the policy has several conflicting grasps to imitate and misses by choosing between or averaging them. Notebook 06 found a wrist-angle spread at the grasp of 21° in `ours_all` vs 5.6° in the teleop demos, and a policy that collapses (57% -> 12%) when its successive predictions are averaged, while the teleop policy does not.

**Experiment:** train ACT, with exactly the settings of every other run, on the episodes of the 5 training demos whose grasp sides lie within 10° of each other (106-116°): `f9115215f9`, `c5f525e833`, `9a63cb1538`, `92e0b86bc6`, `65ee6ec786`. No new data is generated: it is a subset of `ours_all` (about 250 of its 438 episodes). Fewer demos and fewer episodes work *against* this run, so an improvement cannot come from more data.

**Prediction, fixed before running** (success pooled over two evaluations, n = 100; `ours_all` = 57% [47-66]):
* H2 is the main cause -> clearly above `ours_all` (>= 70%), and temporal ensembling no longer collapses (within ~15 points of the default execution).
* H2 is not the main cause -> at or below `ours_all` (<= 60%).
* In between: inconclusive; the next control would be all 10 demos subsampled to the same episode count.

Caveat: the 5 demos are not a random choice; they are also the ones with the most typical grasp side. The data-generation perturbation (±11° along the rim) remains, so the grasp-angle spread in this subset is reduced, not removed (measured below).

**Runtime:** T4, about 2 h (dataset ~15 min, training ~1.2 h, three evaluations ~25 min).

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
# LIBERO needs egl_probe / hf-egl-probe, which are built from source with cmake. pip's isolated build cannot run
# the pip-installed cmake wrapper, and the old egl_probe CMakeLists is rejected by cmake 4: build both first,
# without isolation and with the CMake 3.5 policy floor.
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools", "wheel", "cmake>=3.29,<4.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "egl_probe", "hf-egl-probe"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
DEMOS = "f9115215f9_lefthand,c5f525e833,9a63cb1538_lefthand,92e0b86bc6_lefthand,65ee6ec786"   # grasp sides 106-116°
NAME, SEED, STEPS, BATCH = "ours_consistent", 0, 20000, 32
RUN = f"act_{NAME}_s{SEED}"
from p2p.hub import pull, push

## Dataset: the subset of the generated episodes, through the same writer as every other dataset

In [ ]:
if not os.path.exists(f"{BASE}/generated"):
    pull(HUB_REPO, f"{BASE}/hub", allow_patterns=["raw/generated.tar"])
    !tar -xf {BASE}/hub/raw/generated.tar -C {BASE}
!python -m p2p.data.lerobot_io ours {BASE}/generated {BASE}/lerobot/{NAME} --demos {DEMOS}
!python tools/demo_consistency.py {BASE}/lerobot/{NAME}
push(f"{BASE}/lerobot/{NAME}", HUB_REPO, path_in_repo=f"lerobot/{NAME}")

## Train (same settings as notebook 03) and evaluate: default execution twice, then temporal ensembling

In [ ]:
!python -m p2p.policy.act train {BASE}/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 3 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}

In [ ]:
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")